# Lección 3: Semana 3, cuaderno de grupo

Cuaderno de grupo de la semana 3. El lugar de cada artículo en la red de enlaces y una etiqueta dada. La lista de inicio se escribe en el cuaderno del proyecto.

---
<font size="3">
Christian Cabrera Jojoa<br>
Assistant Research Professor<br>
Department of Computer Science and Technology<br>
University of Cambridge<br>
chc79@cam.ac.uk
</font>

---
**Curso:** Data Mining<br>
**Departamento del curso:** Departamento de Matemáticas y Estadística - Facultad de Ciencias Exactas y Naturales<br>
**Institución del curso:** Universidad de Nariño


## Instrucciones

**Propósito.** Cuaderno de grupo de la semana 3 (Lecciones 5 y 6). Su grupo sigue como equipo interno de una alcaldía, con los artículos de su secretaría. Esta semana produce dos resultados. El cuaderno del proyecto es el que junta las tres semanas y escribe la lista de inicio.

**Esta semana.** Se suman dos resultados: el lugar de cada artículo en la red de enlaces (`outputs/grafo.csv`) y una predicción (`outputs/prediccion.csv`). Esos archivos son una entrada del cuaderno del proyecto. Aquí no se arma la lista.

**Antes de empezar.** Ejecuten primero los cuadernos individuales de las Lecciones 5 y 6. Este cuaderno lo escribe el grupo. Cada paso dice qué código de la Lección 5 o de la Lección 6 adaptar. No peguen sin entender.

**Datos de entrada.** El mismo archivo de su grupo: G1 Agricultura, G2 Salud mental, G3 Medio ambiente, G4 Inclusión Social y G5 Vivienda. La columna `links` trae los enlaces de la entradilla a otros artículos del mismo extracto. Un enlace es una mención en el texto, no una cita. Las categorías las pusieron quienes editan Wikipedia.

**Materiales.** [L5 Grafos](https://colab.research.google.com/github/cabrerac/cabrerac.github.io/blob/gh-pages/assets/notebooks/26-udenar-data-mining/l5-graphs.ipynb) y [L6 Predicción](https://colab.research.google.com/github/cabrerac/cabrerac.github.io/blob/gh-pages/assets/notebooks/26-udenar-data-mining/l6-prediction.ipynb).

**Pregunta del grupo.** ¿Qué lugar ocupa un artículo entre los demás, y la predicción de la Lección 6 cuenta para su secretaría?

**Mapa del cuaderno**

| Ejercicio | Tema | Entregable clave |
|-----------|------|------------------|
| **0** | Configuración | Grupo, etiqueta, datos, parámetros y funciones |
| **1** | Comprensión del negocio | Los resultados de la semana 2 |
| **2** | Comprensión de los datos | Títulos únicos y enlaces dentro del extracto |
| **3** | Preparación de los datos | Los grafos y la tabla de la L6 |
| **4** | Modelado | Grado, PageRank, node2vec y los tres métodos de la L6 |
| **5** | Evaluación | Un artículo por su lugar y si la L6 cuenta |
| **6** | Despliegue | `outputs/grafo.csv`, `outputs/prediccion.csv` y cómo entraría cada resultado |

**Cómo leer cada paso.** Cada paso empieza con **En este paso**, dice qué copiar y dónde está, y termina con una celda **Comprobar**. Ejecuten de arriba abajo. Una Comprobar revisa que exista lo pedido, con su forma. No revisa su respuesta. Si falla, corrijan la celda anterior y vuelvan a ejecutar desde ahí.

---

## Ejercicio 0. Configuración

### Paso 0.1. Grupo, etiqueta y datos

En este paso eligen el grupo, ven la etiqueta de la Lección 6 y descargan el archivo de su secretaría. Suban los archivos de salida de su ZIP de la semana anterior (`outputs/en_vista.csv` y `etiquetas.csv`) antes de seguir. `etiquetas.csv` no se lee aquí, pero va de nuevo en el ZIP. La pregunta y el problema están en el documento de requerimientos.

Cambien solo `GROUP`. La etiqueta de la Lección 6 es `LABEL = "categoria_objetivo"`: 1 si la categoría del artículo (`categoria`) está en `TARGETS[GROUP]`, y 0 si no. `TARGETS` son las mismas categorías del filtro de la semana 2.

In [ ]:
from pathlib import Path

import pandas as pd
import requests

GROUP = "G1"   # cambie a G2, G3, G4 o G5
LABEL = "categoria_objetivo"   # 1 si categoria está en TARGETS[GROUP], 0 si no
TARGETS = {"G1": ["Agrarismo", "Organizaciones agrícolas"], "G2": ["Trastornos mentales", "Psiquiatría"], "G3": ["Conservacionismo"],
           "G4": ["Hambrunas", "Hambre", "Bancos de alimentos"], "G5": ["Hipotecas", "Arrendamiento", "Huelgas de alquileres",
           "Vivienda asequible", "Vivienda pública"]}
EXTRACTS = {"G1": ("Agricultura", "agricultura.csv"), "G2": ("Salud mental", "salud-mental.csv"), "G3": ("Medio ambiente", "medio-ambiente.csv"),
            "G4": ("Inclusión Social", "inclusion-social.csv"), "G5": ("Vivienda", "vivienda.csv")}
SECRETARIAT, FILENAME = EXTRACTS[GROUP]
Path("outputs").mkdir(exist_ok=True)
path = Path("data") / FILENAME
path.parent.mkdir(exist_ok=True)
if not path.is_file():   # descarga una sola vez
    path.write_bytes(requests.get(f"https://cabrerac.github.io/assets/data/26-udenar-data-mining/{FILENAME}", timeout=60).content)
df = pd.read_csv(path, keep_default_na=False).drop_duplicates("title").reset_index(drop=True)
df = df.rename(columns={"label": "categoria"})   # en el extracto la columna se llama label
print(f"Grupo {GROUP}: secretaría de {SECRETARIAT}, {len(df)} artículos sin títulos repetidos. Etiqueta: {LABEL}")

### Paso 0.2. Parámetros, librerías y funciones

En este paso copian los parámetros, las librerías y las funciones de las Lecciones 5 y 6.

| Qué copiar | Dónde está |
|------------|------------|
| Los parámetros de la L5 (`SUPPORT_SHARE`, `SUPPORT_MIN`, `SEED`, `MIN_SUPPORTS`, `HUB_SHARE` a `N_RANDOM_PAIRS`), sin cambiar sus valores. `MIN_SUPPORTS` empieza en 2 y se puede cambiar. Cada grupo fija y justifica este número en su proyecto. 0 o 1 valen si se explica | L5, Parte 1, Paso 2 |
| Los parámetros de la L6 (`FLAGGED_SHARE` a `N_FOLDS` y `HIDDEN_UNITS` a `SPLIT_SEEDS`), sin `LABEL` ni cambiar sus valores | L6, Parte 1, Paso 2 |
| Las importaciones de la L5 (networkx, gensim, random) | L5, Parte 1, Paso 2 |
| Las importaciones de la L6 (sklearn, scipy) | L6, Parte 1, Paso 2 |
| `support_floor` | L2, Parte 1, Paso 2 |
| `hub_threshold` | L5, Parte 1, Paso 2 |
| `n_flagged` y `min_positives` | L6, Parte 1, Paso 2 |
| `PROPERTY_COLUMN`, `LABEL_PROPERTY` y `n_claims_without` | L6, Parte 1, Paso 3 |
| `node2vec_walks` | L5, Parte 4, M3. Las caminatas |

In [ ]:
# SU CÓDIGO. Paso 0.2: parámetros, rutas de la entrega, importaciones, support_floor, hub_threshold, n_flagged, min_positives, n_claims_without, node2vec_walks y vote_of


**Listo cuando:**
- Los parámetros tienen los valores de las Lecciones 5 y 6.
- Las rutas son, en `outputs/`, `en_vista.csv` (semana 2), `grafo.csv` y `prediccion.csv`.
- `vote_of(predicciones)` recibe las predicciones de los 3 métodos y devuelve, por fila, la clase que dicen al menos 2 como texto, o `None` si los tres dicen algo distinto. La etiqueta puede tener varias clases.

**Comprobar:**

En este paso comprobamos que existen las funciones de node2vec y de votación.

In [ ]:
assert callable(node2vec_walks) and vote_of([["a", "b"], ["a", "c"], ["b", "d"]]) == ["a", None]
print("Paso 0.2: OK")

---

## Ejercicio 1. Comprensión del negocio

### Paso 1.1. Los resultados de la semana 2

En este paso leen los resultados de la semana 2 y escriben qué esperan del grafo.
La pregunta y el problema siguen en el documento de requerimientos.

| Qué copiar | Dónde está en L5 |
|------------|------------------|
| Leer los resultados en `week2`, con las tres columnas en verdadero o falso | Parte 1, Paso 5 (aquí el archivo es `outputs/en_vista.csv`) |

In [ ]:
# SU CÓDIGO. Paso 1.1: week2 y graph_expectation


**Listo cuando:**
- `week2` tiene `title`, `has_pattern`, `read_group`, `text_says_read` y `prob_texto`.
- `graph_expectation` (texto, escrito antes de mirar resultados) dice qué artículo creen que será el más enlazado y si una persona nueva debería abrirlo primero.

**Comprobar:**

En este paso comprobamos que leyeron los resultados de la semana 2 y que escribieron qué esperan del grafo.

In [ ]:
assert "prob_texto" in week2 and graph_expectation.strip()
print("Paso 1.1: OK")

---

## Ejercicio 2. Comprensión de los datos

### Paso 2.1. Títulos y enlaces

En este paso revisan los títulos y cómo vienen los enlaces.

| Qué copiar | Dónde está en L5 |
|------------|------------------|
| `titles`, `N` y los títulos repetidos | Parte 2, Paso 1 (aquí `df` ya viene sin repetidos) |
| `listed`, los enlaces escritos | Parte 2, Paso 2 |

In [ ]:
# SU CÓDIGO. Paso 2.1: titles, N y listed


**Listo cuando:**
- `titles` tiene un título por artículo y `listed` una lista de enlaces por artículo.
- Saben cuántos artículos no enlazan a nadie, cuántos no tienen `inception_year` y cuántos tienen `kind = Other`.

**Comprobar:**

En este paso comprobamos que hay un título y una lista de enlaces por artículo.

In [ ]:
assert len(titles) == N == len(listed) == len(df)
print("Paso 2.1: OK")

---

## Ejercicio 3. Preparación de los datos

### Paso 3.1. Los dos grafos y el componente mayor

En este paso arman los dos grafos y el componente mayor.

| Qué copiar | Dónde está en L5 |
|------------|------------------|
| `links_out` y `neighbours` | Parte 3, Paso 1 (use `df` en lugar de `papers`) |
| `directed` y `graph` | Parte 3, Paso 2 |
| `largest`, `adj` y `no_vector` | Parte 3, Paso 3 |

In [ ]:
# SU CÓDIGO. Paso 3.1: links_out, neighbours, directed, graph, largest, adj y no_vector


**Listo cuando:**
- Los dos grafos tienen un nodo por título.
- Cada artículo está en `adj` o en `no_vector`, nunca en los dos.

**Comprobar:**

En este paso comprobamos que el grafo tiene un nodo por artículo y que cada artículo está en `adj` o en `no_vector`.

In [ ]:
assert graph.number_of_nodes() == N and len(adj) + len(no_vector) == N and not set(adj) & set(no_vector)
print("Paso 3.1: OK")

### Paso 3.2. La tabla de la Lección 6

En este paso arman la tabla con etiqueta, el conjunto apartado y la preparación.

| Qué copiar | Dónde está en L6 |
|------------|------------------|
| `attributes` con `kind`, `year`, `has_date`, `has_country`, `has_image` y `n_claims_without` | Parte 3, Paso 1 |
| `train`, `test`, `y_train` y `y_test` con `n_flagged(N)` filas apartadas | Parte 3, Paso 2 |
| `prepare`, el `ColumnTransformer` | Parte 3, Paso 3 |

In [ ]:
# SU CÓDIGO. Paso 3.2: label_of, attributes, NUMBER_COLUMNS, CATEGORY_COLUMNS, y, inputs, train, test, y_train, y_test y prepare


**Listo cuando:**
- `label_of(attributes)` lee la etiqueta solo de su columna, según `LABEL`, y se detiene con un error si falta. Con `"categoria_objetivo"` da 1 si `categoria` está en `TARGETS[GROUP]` y 0 si no. `y = label_of(attributes)`.
- `kind` deja como `Other` los tipos con menos de `support_floor(N)` artículos. Si `LABEL` es `"tiene_imagen"`, `has_image` no es entrada.
- En los números `add_indicator=False`, porque `has_date` ya marca el año faltante.
- `test` tiene `n_flagged(N)` filas y no comparte títulos con `train`.

**Comprobar:**

En este paso comprobamos el tamaño de la etiqueta y del conjunto apartado, y que no comparte filas con el entrenamiento.

In [ ]:
assert len(y) == N and len(test) == n_flagged(N) and not set(train.index) & set(test.index)
print("Paso 3.2: OK")

---

## Ejercicio 4. Modelado

### Paso 4.1. Grado y PageRank

En este paso calculan el grado, el PageRank y los artículos con PageRank alto.

| Qué copiar | Dónde está en L5 |
|------------|------------------|
| `degree`, `hubs` con `hub_threshold(N)` y `top_degree` | Parte 4, M1. El grado y la alerta de nodo central |
| `rank` con `nx.pagerank(directed, alpha=DAMPING)` y `top_rank` | Parte 4, M2. PageRank con networkx |
| `score`, PageRank redondeado a 6 decimales, y `high_rank` | Parte 4, M2. Los artículos con PageRank alto |

In [ ]:
# SU CÓDIGO. Paso 4.1: degree, hubs, top_degree, rank, top_rank, score y high_rank


**Listo cuando:**
- Un artículo con PageRank alto tiene enlaces, está en el cuarto superior por PageRank entre los artículos con enlaces y **no** es nodo central.
- Un nodo central (`hubs`) es una alerta, no una razón para leerlo.

**Comprobar:**

En este paso comprobamos que hay al menos un artículo con PageRank alto y que ninguno es nodo central.

In [ ]:
assert len(high_rank) >= 1 and not set(high_rank) & set(hubs)
print("Paso 4.1: OK")

### Paso 4.2. node2vec

En este paso entrenan node2vec con tres semillas y guardan los vecinos de cada artículo.

| Qué copiar | Dónde está en L5 |
|------------|------------------|
| Las caminatas y `Word2Vec` con `workers=1` | Parte 4, M3. Los vectores de node2vec |
| `neighbours_by_seed`, un diccionario por semilla con los 5 vecinos de cada artículo de `adj` | Parte 5, Paso 3 |

In [ ]:
# SU CÓDIGO. Paso 4.2: neighbours_by_seed y model


**Listo cuando:**
- Hay 3 diccionarios de vecinos, uno por semilla de `N2V_SEEDS`.
- `model` es el de la primera semilla, con un vector de `NODE_DIMENSIONS` números por artículo de `adj`.

**Comprobar:**

En este paso comprobamos que hay tres semillas de vecinos y un vector por artículo de `adj`.

In [ ]:
assert len(neighbours_by_seed) == 3 and model.wv.vectors.shape == (len(adj), NODE_DIMENSIONS)
print("Paso 4.2: OK")

### Paso 4.3. Los tres métodos de la Lección 6

En este paso entrenan los tres métodos de la Lección 6 y predicen las filas apartadas.

| Qué copiar | Dónde está en L6 |
|------------|------------------|
| `lr` | Parte 4, M1. Regresión logística |
| `forest` | Parte 4, M2. Bosque aleatorio |
| `network` | Parte 4, M3. Una capa oculta |
| `models` con `make_pipeline(prepare, ...)` y `test_predictions` | Parte 6, Paso 1 |

In [ ]:
# SU CÓDIGO. Paso 4.3: lr, forest, network, models y test_predictions


**Listo cuando:**
- Los tres métodos se entrenan con `train`, y `test_predictions` tiene tres listas, una predicción por fila apartada.

**Comprobar:**

En este paso comprobamos que hay tres modelos y una predicción por fila apartada.

In [ ]:
assert len(models) == 3 and all(len(p) == len(test) for p in test_predictions)
print("Paso 4.3: OK")

---

## Ejercicio 5. Evaluación

### Paso 5.1. Un artículo descrito por su lugar

En este paso miden si los vecinos se repiten y describen un artículo por su lugar en la red.

| Qué copiar | Dónde está en L5 |
|------------|------------------|
| `overlap`, la parte de los 5 vecinos que se repite entre semillas | Parte 5, Paso 3 |
| `near` y `far`, el Jaccard de categorías con vecinos y con pares al azar | Parte 5, Paso 4 (aquí las categorías están en `categories`) |
| `MY_ARTICLE` y su descripción por grado, PageRank y vecinos | Parte 5, Paso 2 |

In [ ]:
# SU CÓDIGO. Paso 5.1: overlap, near, far, MY_ARTICLE y place_reading


**Listo cuando:**
- `MY_ARTICLE` es un título del componente mayor y no es nodo central. Si no tiene vector, lo dicen.
- `place_reading` (texto) dice qué dice su lugar y por qué ser central solo porque enlaza a muchos no es razón para leerlo.

**Comprobar:**

En este paso comprobamos su artículo, la parte de vecinos que se repite y que escribieron su lectura.

In [ ]:
assert MY_ARTICLE in titles and MY_ARTICLE not in hubs and 0 <= overlap <= 1 and place_reading.strip()
print("Paso 5.1: OK")

### Paso 5.2. ¿Cuenta la predicción?

En este paso deciden si la predicción le gana a la clase mayor y si cuenta en la lista.

| Qué copiar | Dónde está en L6 |
|------------|------------------|
| `folds` y `train_predictions`, las predicciones fuera del pliegue | Parte 5, Paso 3 |
| `evaluation`, exactitud y F1 macro de la clase mayor y de los tres métodos | Parte 5, Pasos 1 y 3 |
| `majority` y `l6_counts` | Parte 5, Paso 4 |
| `seeds_holding` con `SPLIT_SEEDS` y `clone(m)` | Parte 5, Paso 6 |

In [ ]:
# SU CÓDIGO. Paso 5.2: folds, train_predictions, evaluation, majority, l6_counts, seeds_holding y l6_finding


**Listo cuando:**
- La clase mayor es `y_test.value_counts(normalize=True).max()`.
- `l6_counts` dice si la predicción entra en la lista. Es `True` solo si cada uno de los tres métodos acierta en las filas apartadas más que la clase mayor, por un margen de `MARGIN` (0,02, una o dos filas de unas 60, como en la Lección 6). Si es `False`, la predicción no apoya su lista. Es un resultado normal y válido, no un error: lo informan en el proyecto y explican por qué (los métodos no le ganan a la clase mayor por más de `MARGIN`).
- `l6_finding` (texto) da la clase mayor, la exactitud de los tres métodos, `l6_counts` y en cuántas de las 5 particiones se sostiene, y qué significa para la lista.

**Comprobar:**

En este paso comprobamos que `l6_counts` es verdadero o falso y que escribieron lo que encontraron.

In [ ]:
assert isinstance(l6_counts, bool) and l6_finding.strip()
print("Paso 5.2: OK")

---

## Ejercicio 6. Despliegue

### Paso 6.1. Guardar el grafo y la predicción

En este paso guardan los dos resultados de esta semana y escriben cómo entraría cada uno en la lista.
La lista se escribe en el cuaderno del proyecto.

| Archivo | Columnas | Dónde está |
|---------|----------|------------|
| `outputs/grafo.csv` | `title`, `grado`, `pagerank`, `aislado`, `alerta_centro` y `graph_support`. Pueden sumar `sin_vector_n2v` y `vecinos_n2v` | L5, Parte 6, Paso 2 |
| `outputs/prediccion.csv` | `title`, `etiqueta`, `apartada`, `clase_votada` y `prediction` | L6, Parte 6, Paso 6 |

In [ ]:
# SU CÓDIGO. Paso 6.1: outputs/grafo.csv, outputs/prediccion.csv, GRAPH_USE y PREDICTION_USE


**Listo cuando:**
- `graph_support` es verdadero si el artículo está en `high_rank`.
- `prediction` es verdadero si la clase votada es `"1"`.
- `alerta_centro` queda en el archivo del grafo. No es un resultado a favor.
- `GRAPH_USE` y `PREDICTION_USE` (texto) dicen cómo entraría cada resultado y qué hecho lo cambiaría.

**Comprobar:**

En este paso comprobamos que escribieron el uso del grafo y de la predicción, y que se guardaron los dos archivos.

In [ ]:
assert GRAPH_USE.strip() and PREDICTION_USE.strip()
grafo = pd.read_csv("outputs/grafo.csv")
prediccion = pd.read_csv("outputs/prediccion.csv")
assert {"title", "graph_support", "alerta_centro"} <= set(grafo.columns)
assert {"title", "prediction"} <= set(prediccion.columns)
print("Paso 6.1: OK")

## Registro de contribución

En este paso cada integrante anota su tarea concreta y el grupo dice quién consolida el ZIP.

In [ ]:
# SU CÓDIGO. Registro: contribution_log, una línea por integrante y quién consolida el ZIP


**Listo cuando:**
- `contribution_log` (texto) tiene una línea con una tarea concreta por integrante y dice quién consolidó el ZIP.

## Para la reflexión individual

Use la [plantilla de la reflexión de la semana 3](https://cabrerac.github.io/assets/documents/26-udenar-data-mining/reflection-week-3-template.docx) y mantenga sus títulos. Extensión: entre 700 y 1000 palabras. Por debajo de 500 palabras el texto queda corto.

| Título | Qué escribir |
|--------|--------------|
| **Metadatos** | Nombre, grupo, semana 3 (L5 + L6), conteo aproximado de palabras y la lectura que cita. |
| **Proceso** | Qué hizo el grupo en el cuaderno de la semana 3. Diga cuál fue su contribución. |
| **Decisión** | Elija un artículo y descríbalo por su lugar en la red. Diga si la predicción de esta semana cuenta para ese artículo y por qué. Un artículo en el centro solo porque enlaza con muchos es una alerta, no una razón. |
| **Métodos** | Explique cuáles fueron los métodos utilizados en esta semana (grado, PageRank, node2vec y los modelos de predicción) y cómo se comparan entre ellos. |
| **Ética** | Nombre una elección técnica de esta semana. Diga a quién ayuda o perjudica, o cómo se podría usar mal un resultado de esta semana. Relacione el punto con Page, Brin, Motwani y Winograd (1998) o con LeCun, Bengio y Hinton (2015). |

## Entrega

El ZIP se entrega el martes 1 dic 2026, 23:59 (hora de Colombia), con el cuaderno ejecutado y renombrado.
La reflexión individual se entrega el miércoles 2 dic 2026, 23:59 (hora de Colombia).

| Archivo del ZIP `week-3-<group_id>.zip` | Descripción |
|-----------------------------------------|-------------|
| `week-3-group-<group_id>.ipynb` | Este cuaderno ejecutado |
| `etiquetas.csv` | Su conjunto etiquetado de la semana 2 |
| `outputs/grafo.csv` | El lugar de cada artículo en la red, con `graph_support` y `alerta_centro` |
| `outputs/prediccion.csv` | La clase votada y el resultado de la predicción |

Estos archivos son la tercera iteración.
El cuaderno del proyecto los lee, junto con los de las semanas 1 y 2, y escribe la lista de inicio.